# HybridTCN Detailed Backtest

Loads the best model and hyperparameters saved by **`hybridtcn_cl_ho_rb_optuna.ipynb`**
and produces a full trading backtest covering:

| Section | Metrics |
|---|---|
| Portfolio summary | Sharpe, Sortino, Calmar (annualised), profit factor, max-DD |
| Position evolution | Daily avg weight across time, rolling exposure, per-ticker heatmap |
| Calibration | Position vs realized-return quantile, position–return scatter |
| Trade-level | Avg profit / trade, holding duration, long vs short breakdown |
| Rolling | 30-day rolling Sharpe, rolling drawdown |
| Calendar | Monthly PnL heatmap |
| **Branch Attribution** | Daily rolling 3-branch weights (TCN + seq + spatial) vs price vs PnL |
| **Feature Importance** | Gradient-based tech feature and sequential feature attribution |

Run the training notebook first to generate the checkpoint and `best_params.json`.

## 1. Config

In [ ]:
from pathlib import Path
import numpy as np

# -- Must match training notebook ----------------------------------------------
TICKERS                 = ['CL', 'HO', 'RB']
USE_PTP                 = False
USE_FUSED_SPATIAL       = True
SPATIAL_ENCODER         = 'fused' if USE_FUSED_SPATIAL else 'separate'
SPATIAL_LOOKBACK_BARS   = 8
BAR_MINUTES             = 5
TARGET_HORIZON_MINUTES  = 60
SAMPLE_SESSION          = 'overlap'
SAMPLE_SESSION_START    = '06:00'
SAMPLE_SESSION_END      = '13:00'
SAMPLE_STRIDE           = 12
AE_WINDOW               = 21
F_AE                    = 4
ARTIFACT_STEM           = '_'.join(TICKERS).lower()

IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT    = Path('/content/drive/MyDrive/model_data')
else:
    DATA_ROOT    = Path(r'F:\Upload\s3\model_data')

RESULTS_PATH = Path('/content/drive/MyDrive/results/hybrid_tcn_cl_ho_rb')
BACKTEST_PATH = RESULTS_PATH
BACKTEST_PATH.mkdir(parents=True, exist_ok=True)

prefix      = f'{ARTIFACT_STEM}_hybridtcn_optuna'
params_path = RESULTS_PATH / f'{prefix}_best_params.json'
study_path  = RESULTS_PATH / f'{prefix}_study.pkl'
ckpt_path   = RESULTS_PATH / f'{prefix}_best_model.pth'

required_files = [params_path, ckpt_path]
required_files.extend(DATA_ROOT / ticker / 'intraday.csv' for ticker in TICKERS)
required_files.extend(DATA_ROOT / ticker / f'{ticker}_numbars.npz' for ticker in TICKERS)
required_files.extend(DATA_ROOT / ticker / 'vpin.parquet' for ticker in TICKERS)
missing_required_files = [path for path in required_files if not path.exists()]

optional_files = [study_path]
optional_files.extend(DATA_ROOT / ticker / 'rasterized.npz' for ticker in TICKERS)
optional_files.extend(DATA_ROOT / ticker / 'profiles.npz' for ticker in TICKERS)
missing_optional_files = [path for path in optional_files if not path.exists()]

# -- Backtest params ------------------------------------------------------------
BACKTEST_BATCH_SIZE = 256
TRADE_THRESH        = 0.05
TC_COST_BPS         = 0.5
TC_COST             = TC_COST_BPS * 1e-4

# Session: 06:00-13:00 overlap -> 420 min
SESSION_MINUTES = 420
BARS_PER_DAY   = int(SESSION_MINUTES / TARGET_HORIZON_MINUTES)
BARS_PER_YEAR  = 252 * BARS_PER_DAY
ANNUALIZATION  = np.sqrt(BARS_PER_YEAR)

print(f'prefix            : {prefix}')
print(f'RESULTS_PATH      : {RESULTS_PATH}')
print(f'DATA_ROOT         : {DATA_ROOT}')
print(f'params_path       : {params_path}')
print(f'ckpt_path         : {ckpt_path}')
print(f'SPATIAL_ENCODER   : {SPATIAL_ENCODER}')
print(f'SPATIAL_LOOKBACK  : {SPATIAL_LOOKBACK_BARS}')
print(f'BARS_PER_DAY      : {BARS_PER_DAY}   ANNUALIZATION: {ANNUALIZATION:.2f}')

if missing_required_files:
    print('\nMissing required files:')
    for path in missing_required_files:
        print(f'  - {path}')
else:
    print('\nAll required files are present.')

if missing_optional_files:
    print('\nMissing optional files:')
    for path in missing_optional_files:
        print(f'  - {path}')

## 2. Imports

In [ ]:
import gc, json, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
from torch.utils.data import DataLoader
warnings.filterwarnings('ignore')

from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    build_v3_loaders,
    unpack_v3_batch,
    v3_collate_fn,
)
from CTAFlow.models.prep.intraday_continuous import SessionSpec
from CTAFlow.models.deep_learning.multi_branch.tcn import HybridTCN
from CTAFlow.models.deep_learning.multi_branch.tft import (
    StatefulMMTFv3Core,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 3. Load Saved Artefacts

In [ ]:
# -- Best hyperparameters ------------------------------------------------------
with open(params_path) as f:
    best = json.load(f)

print('Best hyperparameters loaded:')
for k, v in sorted(best.items()):
    print(f'  {k:35s}: {v}')

# -- Optional Optuna study for trial history -----------------------------------
import joblib
study = None
if study_path.exists():
    study = joblib.load(study_path)
    print(f'\nOptuna study loaded  - {len(study.trials)} trials')
    print(f'  Best value : {study.best_value:.6f}')
    print(f'  Best trial : #{study.best_trial.number}')
else:
    print('\nNo Optuna study found; skipping trial history.')

## 4. Reconstruct Data Prep & Model

In [ ]:
# -- V3ContinuousPrep ----------------------------------------------------------
if missing_required_files:
    missing_text = '\n'.join(f'  - {path}' for path in missing_required_files)
    raise FileNotFoundError(
        'Add the missing required files before running the backtest:\n'
        f'{missing_text}'
    )

prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec('custom', '06:00', '13:00')],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)
dims = prep.get_dims()
F_TECH               = dims['f_tech']
F_SEQ                = dims['f_seq']
NUMBARS_CHANNELS     = dims['numbars_channels']
VPIN_TIME            = dims['vpin_time']
VPIN_CHANNELS        = dims['vpin_channels']
VPIN_BINS            = dims['vpin_bins']
FUSED_SPATIAL_CHANNELS = NUMBARS_CHANNELS + 3
FUSED_SPATIAL_BINS     = prep.numbars_bar_shape[1]
print('Dims:', dims)
print(f'Fused spatial shape: ({FUSED_SPATIAL_CHANNELS}, {FUSED_SPATIAL_BINS})')

for ticker in TICKERS:
    assert prep.registry[ticker].numbars_df is not None, f'Missing NumberBars for {ticker}'

# -- Reconstruct HybridTCN -----------------------------------------------------
tcn_depth = int(best.get('tcn_depth', 4))
d_model = int(best['d_model'])

base_model = HybridTCN(
    f_tech=F_TECH, f_seq=F_SEQ, f_ae=F_AE,
    d_latent=best['d_latent'], d_ae_hidden=best['d_ae_hidden'],
    kl_weight=best['kl_weight'], recon_weight=best['recon_weight'],
    numbars_channels=NUMBARS_CHANNELS, vpin_channels=VPIN_CHANNELS,
    vpin_bins=VPIN_BINS, vpin_time=VPIN_TIME,
    n_tickers=prep.n_tickers,
    n_asset_classes=prep.n_asset_classes,
    n_asset_subclasses=prep.n_asset_subclasses,
    d_model=d_model, d_static_emb=best['d_static_emb'],
    tcn_channels=[d_model] * tcn_depth,
    kernel_size=best['kernel_size'],
    spatial_encoder=SPATIAL_ENCODER,
    seq_layers=best['seq_layers'], seq_nheads=best['seq_nheads'],
    regime_gate=True, regime_floor=best['regime_floor'],
    dropout=best['dropout'],
)

model = StatefulMMTFv3Core(
    base_model=base_model,
    n_tickers=prep.n_tickers,
    quantile_head=False,
    state_hidden_dim=best.get('state_hidden_dim', 16),
    state_momentum=best.get('state_momentum', 0.9),
    update_on_eval=True,
).to(device)

# -- Load checkpoint ------------------------------------------------------------
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()
print(f'\nCheckpoint loaded from {ckpt_path}')
print(f'  Trained metrics: {ckpt.get("final_metrics", ckpt.get("metrics", {}))}')
n_params = sum(p.numel() for p in model.parameters())
print(f'  Parameters     : {n_params:,}')
print(f'  TCN receptive field: {base_model.tcn.receptive_field}')

## 5. Build Date-Aware Dataset & OOS-Only Backtest

Samples are built over the full date range, then filtered to **OOS only**
(last 25% by date) for the backtest.

In [ ]:
tech_lookback     = int(best['tech_lookback'])
seq_lookback      = int(best['seq_lookback'])
numbars_lookback  = int(best.get('numbars_lookback', SPATIAL_LOOKBACK_BARS))

# Build all samples with BEST lookbacks (not MAX — saves memory)
all_samples = prep.build_samples(
    tech_lookback=tech_lookback,
    seq_lookback_bars=seq_lookback,
    numbars_lookback=numbars_lookback,
    use_fused_spatial=USE_FUSED_SPATIAL,
    stride=SAMPLE_STRIDE,
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
)
print(f'Total samples (all): {len(all_samples):,}')
print(f'Spatial lookback bars: {numbars_lookback}')

# -- Free prep internals immediately (biggest RAM saver) -----------------------
_seq_cols_saved = None
for tk in TICKERS:
    if tk in prep._seq_vpin and len(prep._seq_vpin[tk].columns) > 0:
        _seq_cols_saved = list(prep._seq_vpin[tk].columns)
        break
_tech_cols_saved = list(prep._tech_feature_cols)

for ticker in list(prep._tech_dfs.keys()):
    prep._tech_dfs[ticker] = None
for ticker in list(prep._seq_vpin.keys()):
    prep._seq_vpin[ticker] = None
for ticker in list(prep._vpin_spatial.keys()):
    prep._vpin_spatial[ticker] = None
for ticker in list(prep._numbars_ts.keys()):
    prep._numbars_ts[ticker] = (np.array([], dtype='datetime64[ns]'),
                                 np.empty((0, 4, 32), dtype=np.float32))
for ticker in list(prep._ae_features.keys()):
    prep._ae_features[ticker] = {}
for attr in ('_rasters', '_profiles'):
    if hasattr(prep, attr):
        for ticker in list(getattr(prep, attr).keys()):
            getattr(prep, attr)[ticker] = {}

gc.collect()
print(f'Prep internals freed. RAM reclaimed.')

# -- Val split: last 25% by DATE -----------------------------------------------
VAL_RATIO   = 0.25
all_dates    = [s['date'] for s in all_samples]
unique_dates = sorted(set(all_dates))
val_date_idx = int(len(unique_dates) * (1 - VAL_RATIO))
VAL_START_DATE = unique_dates[val_date_idx]
print(f'VAL_START_DATE    : {VAL_START_DATE}')

# -- Filter to OOS samples only ------------------------------------------------
oos_samples = [s for s in all_samples if s['date'] >= VAL_START_DATE]
del all_samples
gc.collect()

print(f'OOS samples       : {len(oos_samples):,}  ({VAL_START_DATE} -> {oos_samples[-1]["date"]})')

sample_dates   = [s['date']   for s in oos_samples]
sample_tickers = [s['ticker'] for s in oos_samples]

for tk in TICKERS:
    n_tk = sum(1 for t in sample_tickers if t == tk)
    print(f'  {tk}: {n_tk:,} OOS samples')

# -- DataLoader (no shuffle) ---------------------------------------------------
dataset     = V3ContinuousDataset(
    oos_samples,
    fused_tail_shape=(FUSED_SPATIAL_CHANNELS, FUSED_SPATIAL_BINS),
)
full_loader = DataLoader(
    dataset, batch_size=BACKTEST_BATCH_SIZE,
    shuffle=False, collate_fn=v3_collate_fn, num_workers=0,
)
print(f'DataLoader  : {len(full_loader)} batches of up to {BACKTEST_BATCH_SIZE}')

## 6. Inference Pass (with Branch Weight Collection)

Captures per-sample weights for all three branches:
1. **TCN (Technical)** — L2 norm of `z_temporal` from TCN backbone
2. **Sequential (VPIN)** — regime-gated sequential weight
3. **Spatial (NB+VPIN)** — regime-gated spatial weight

In [ ]:
bt_pos_list, bt_ret_list, bt_tid_list = [], [], []
gate_seq_list, gate_spatial_list, regime_weight_list = [], [], []

# -- Hook to capture per-sample regime gate weights ----------------------------
_captured_gates = []

def _regime_fusion_hook(module, inputs, outputs):
    """Capture (z_fused, gate_seq, gate_spatial) from RegimeGatedFusion."""
    _z_fused, g_seq, g_spatial = outputs
    _captured_gates.append((g_seq.detach().cpu().numpy(),
                            g_spatial.detach().cpu().numpy()))

_captured_regime_w = []

def _regime_scaler_hook(module, inputs, outputs):
    """Capture regime scaler sigmoid output."""
    _captured_regime_w.append(outputs.detach().cpu().numpy())

# -- Hook to capture TCN backbone output norm (z_temporal magnitude) -----------
_captured_tcn_norm = []
_captured_seq_norm = []
_captured_spatial_norm = []

def _tcn_hook(module, inputs, outputs):
    """Capture TCN backbone output (B, C, L) -> L2 norm of last timestep."""
    # outputs is (B, C, L); we take [:, :, -1] norm as proxy for TCN activation
    last_step = outputs[:, :, -1]  # (B, C)
    _captured_tcn_norm.append(last_step.detach().norm(dim=-1).cpu().numpy())  # (B,)

def _seq_encoder_hook(module, inputs, outputs):
    """Capture sequential encoder output norm."""
    _captured_seq_norm.append(outputs.detach().norm(dim=-1).cpu().numpy())  # (B,)

def _spatial_encoder_hook(module, inputs, outputs):
    """Capture spatial encoder output norm."""
    _captured_spatial_norm.append(outputs.detach().norm(dim=-1).cpu().numpy())  # (B,)

hook_fusion  = model.base_model.regime_fusion.register_forward_hook(_regime_fusion_hook)
hook_regime  = model.base_model.regime_scaler.register_forward_hook(_regime_scaler_hook)
hook_tcn     = model.base_model.tcn.register_forward_hook(_tcn_hook)
hook_seq_enc = model.base_model.seq_encoder.register_forward_hook(_seq_encoder_hook)

# Spatial encoder hook (fused or separate)
if model.base_model.fused_spatial_encoder is not None:
    hook_spatial = model.base_model.fused_spatial_encoder.register_forward_hook(_spatial_encoder_hook)
elif model.base_model.spatial_fuse is not None:
    hook_spatial = model.base_model.spatial_fuse.register_forward_hook(_spatial_encoder_hook)
else:
    hook_spatial = None

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

with torch.no_grad():
    for batch in full_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        out = model(**inputs, return_ae_losses=True)
        position, _ae_losses = out

        bt_pos_list.append(position.view(-1).cpu())
        bt_ret_list.append(targets.view(-1).float().cpu())
        bt_tid_list.append(inputs['ticker_id'].view(-1).cpu())

hook_fusion.remove()
hook_regime.remove()
hook_tcn.remove()
hook_seq_enc.remove()
if hook_spatial is not None:
    hook_spatial.remove()

bt_pos = torch.cat(bt_pos_list).numpy()
bt_ret = torch.cat(bt_ret_list).numpy()
bt_tid = torch.cat(bt_tid_list).numpy()

# Unpack captured gates
gate_seq_arr     = np.concatenate([g[0] for g in _captured_gates], axis=0).squeeze(-1)   # (N,)
gate_spatial_arr = np.concatenate([g[1] for g in _captured_gates], axis=0).squeeze(-1)   # (N,)
regime_w_arr     = np.concatenate(_captured_regime_w, axis=0).squeeze(-1)                # (N,)
tcn_norm_arr     = np.concatenate(_captured_tcn_norm, axis=0)                            # (N,)
seq_norm_arr     = np.concatenate(_captured_seq_norm, axis=0)                            # (N,)
spatial_norm_arr = np.concatenate(_captured_spatial_norm, axis=0) if _captured_spatial_norm else np.zeros_like(tcn_norm_arr)

# -- Compute relative branch contribution (softmax over L2 norms) -------------
branch_norms = np.stack([tcn_norm_arr, seq_norm_arr, spatial_norm_arr], axis=-1)  # (N, 3)
# Softmax to get relative contribution
branch_norms_exp = np.exp(branch_norms - branch_norms.max(axis=-1, keepdims=True))
branch_weights_3 = branch_norms_exp / (branch_norms_exp.sum(axis=-1, keepdims=True) + 1e-8)  # (N, 3)

id_to_ticker = {meta.ticker_id: t for t, meta in prep.registry.items()}
print(f'Inference done: {len(bt_pos):,} samples')
print(f'Position range: [{bt_pos.min():.4f}, {bt_pos.max():.4f}]  mean={bt_pos.mean():.4f}')
print(f'Gate seq mean={gate_seq_arr.mean():.4f}  Gate spatial mean={gate_spatial_arr.mean():.4f}')
print(f'Regime weight mean={regime_w_arr.mean():.4f}')
print(f'Branch norms (TCN/Seq/Spatial): '
      f'{tcn_norm_arr.mean():.3f} / {seq_norm_arr.mean():.3f} / {spatial_norm_arr.mean():.3f}')
print(f'Relative branch weights (TCN/Seq/Spatial): '
      f'{branch_weights_3[:, 0].mean():.3f} / {branch_weights_3[:, 1].mean():.3f} / {branch_weights_3[:, 2].mean():.3f}')

## 7. Backtest DataFrame

In [ ]:
bt = pd.DataFrame({
    'date'        : sample_dates,
    'ticker'      : sample_tickers,
    'ticker_id'   : bt_tid.astype(int),
    'position'    : bt_pos,
    'fwd_return'  : bt_ret,
    'gate_seq'    : gate_seq_arr,
    'gate_spatial' : gate_spatial_arr,
    'regime_weight': regime_w_arr,
    'w_tcn'       : branch_weights_3[:, 0],
    'w_seq'       : branch_weights_3[:, 1],
    'w_spatial'   : branch_weights_3[:, 2],
})

bt['date']         = pd.to_datetime(bt['date'])
bt['strategy_ret'] = bt['position'] * bt['fwd_return']
bt['active']       = bt['position'].abs() > TRADE_THRESH

# -- Transaction cost: charge TC_COST on every direction flip -------------------
bt = bt.sort_values(['ticker', 'date']).reset_index(drop=True)
signs           = np.sign(bt['position'].values)
ticker_boundary = (bt['ticker'] != bt['ticker'].shift(1)).values
sign_flip       = np.concatenate([[False], np.diff(signs) != 0])
sign_flip[ticker_boundary] = False

bt['tc_cost']         = np.where(sign_flip, TC_COST, 0.0)
bt['strategy_ret_tc'] = bt['strategy_ret'] - bt['tc_cost']
bt['is_trade']        = sign_flip

# -- Cumulative PnL per ticker -------------------------------------------------
for col in ('strategy_ret', 'strategy_ret_tc'):
    bt[f'cum_{col}'] = bt.groupby('ticker')[col].cumsum()

print(f'OOS backtest: {len(bt):,} samples, {bt["date"].min().date()} -> {bt["date"].max().date()}')
print(f'Columns: {list(bt.columns)}')
print(bt.head(3).to_string())

## 8. Portfolio-Level Metrics

In [ ]:
def _metrics(df, label, col='strategy_ret'):
    """Compute trading metrics for a slice of the backtest DataFrame."""
    sr     = df[col].values
    pos    = df['position'].values
    ret    = df['fwd_return'].values
    active = df['active'].values

    cum     = np.cumsum(sr)
    mean_sr = sr.mean()
    std_sr  = sr.std() + 1e-8
    neg     = sr[sr < 0]
    dside   = float(np.sqrt((neg**2).mean())) if len(neg) > 0 else 1e-8
    gp      = sr[sr > 0].sum()
    gl      = np.abs(sr[sr < 0]).sum() + 1e-9
    run_max = np.maximum.accumulate(cum)
    mdd     = float((run_max - cum).max())

    correct = ((pos > 0) & (ret > 0)) | ((pos < 0) & (ret < 0))
    dir_acc = (correct & active).sum() / max(active.sum(), 1)
    win_rate = ((sr[active] > 0).mean() * 100) if active.sum() > 0 else 0.0

    sharpe_ann  = (mean_sr / std_sr) * ANNUALIZATION
    sortino_ann = (mean_sr / (dside + 1e-8)) * ANNUALIZATION
    annual_ret  = mean_sr * BARS_PER_YEAR
    calmar      = annual_ret / (mdd + 1e-9)

    n_trades = int(df['is_trade'].sum())
    avg_trade = df.loc[df['is_trade'], col].mean() if n_trades > 0 else 0.0

    return {
        'Label'           : label,
        'N Samples'       : len(sr),
        'Net PnL'         : round(float(sr.sum()), 5),
        'Ann. Return'     : round(float(annual_ret), 5),
        'Sharpe (ann)'    : round(float(sharpe_ann), 3),
        'Sortino (ann)'   : round(float(sortino_ann), 3),
        'Calmar'          : round(float(calmar), 3),
        'Win Rate (%)'    : round(float(win_rate), 2),
        'Dir Acc (%)'     : round(float(dir_acc * 100), 2),
        'Profit Factor'   : round(float(gp / gl), 4),
        'Max Drawdown'    : round(mdd, 5),
        '# Trades'        : n_trades,
        'Avg Trade PnL'   : round(float(avg_trade), 6),
        'Avg |Position|'  : round(float(np.abs(pos).mean()), 4),
        'Active Bars (%)' : round(float(active.mean() * 100), 2),
    }

# -- Build table (OOS only) ---------------------------------------------------
rows = []

# Gross metrics
rows.append(_metrics(bt, 'ALL - OOS (gross)'))
for tid in sorted(id_to_ticker):
    tk = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk]
    if len(df_tk) == 0: continue
    rows.append(_metrics(df_tk, f'{tk} - OOS (gross)'))

rows.append({k: '------' if k != 'Label' else '----------------------' for k in rows[0]})

# TC-adjusted metrics
rows.append(_metrics(bt, 'ALL - OOS (TC adj)', col='strategy_ret_tc'))
for tid in sorted(id_to_ticker):
    tk = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk]
    if len(df_tk) == 0: continue
    rows.append(_metrics(df_tk, f'{tk} - OOS (TC adj)', col='strategy_ret_tc'))

df_summary = pd.DataFrame(rows).set_index('Label')
print('=' * 100)
print(f'OOS BACKTEST SUMMARY - HybridTCN Best Model  (from {VAL_START_DATE})')
print('=' * 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print(df_summary.to_string())

## 9. Position (Weight) Evolution Over Time

In [ ]:
daily_pos = (
    bt.groupby(['date', 'ticker'])['position']
    .mean()
    .unstack('ticker')
    .fillna(0)
)
daily_pos['COMBINED'] = daily_pos.mean(axis=1)
roll5 = daily_pos.rolling(5, min_periods=1).mean()

palette = {'CL': '#2980b9', 'HO': '#e67e22', 'RB': '#27ae60', 'COMBINED': 'black'}

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# 1. Raw daily position per ticker
ax = axes[0]
for col in [c for c in daily_pos.columns if c != 'COMBINED']:
    ax.plot(daily_pos.index, daily_pos[col], alpha=0.5, lw=0.8,
            color=palette.get(col), label=col)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Daily Average Position by Ticker (OOS)')
ax.set_ylabel('Avg Position')
ax.legend()
ax.grid(True, alpha=0.25)

# 2. Rolling 5-day
ax = axes[1]
for col in daily_pos.columns:
    ax.plot(roll5.index, roll5[col], lw=1.5,
            color=palette.get(col, 'gray'), label=col)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Rolling 5-Day Avg Position (OOS)')
ax.set_ylabel('Avg Position')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# 3. Exposure heatmap (ticker x month)
ax = axes[2]
hm_data = (
    bt.assign(ym=bt['date'].dt.to_period('M'))
    .groupby(['ym', 'ticker'])['position']
    .apply(lambda x: x.abs().mean())
    .unstack('ticker')
)
hm_data.index = hm_data.index.astype(str)
im = ax.imshow(hm_data.T.values, aspect='auto', cmap='RdYlGn',
               vmin=0, vmax=hm_data.values.max())
ax.set_xticks(range(len(hm_data.index)))
ax.set_xticklabels(hm_data.index, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(hm_data.columns)))
ax.set_yticklabels(hm_data.columns)
ax.set_title('Monthly Avg Absolute Position (Exposure) Heatmap')
plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)

plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_position_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Calibration - Position vs Realized Return Quantile

In [ ]:
N_QUANTILES = 10

fig, axes = plt.subplots(2, len(TICKERS) + 1, figsize=(7 * (len(TICKERS) + 1), 10))

for col_idx, (label, df_slice) in enumerate(
    [('ALL', bt)] + [(tk, bt[bt['ticker'] == tk]) for tk in sorted(id_to_ticker.values())]
):
    df_q = df_slice.copy()
    df_q['ret_q'] = pd.qcut(df_q['fwd_return'], q=N_QUANTILES, labels=False)

    agg = df_q.groupby('ret_q').agg(
        avg_position=('position', 'mean'),
        avg_strat_ret=('strategy_ret', 'mean'),
        avg_fwd_ret=('fwd_return', 'mean'),
        count=('position', 'count'),
    ).reset_index()

    q_labels = [f'D{i+1}' for i in range(N_QUANTILES)]
    cmap_vals = plt.cm.RdYlGn(np.linspace(0, 1, N_QUANTILES))

    ax = axes[0, col_idx]
    ax.bar(q_labels, agg['avg_position'].values, color=cmap_vals, alpha=0.9)
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{label} - Avg Position by Return Decile')
    ax.set_xlabel('Return Decile (D1=worst, D10=best)')
    ax.set_ylabel('Avg Position')
    ax.grid(True, alpha=0.25, axis='y')

    ax = axes[1, col_idx]
    ax.bar(q_labels, agg['avg_strat_ret'].values, color=cmap_vals, alpha=0.9)
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{label} - Avg Strategy Return by Decile')
    ax.set_xlabel('Return Decile')
    ax.set_ylabel('Avg Strategy Return')
    ax.grid(True, alpha=0.25, axis='y')

plt.suptitle('Position Calibration Across Realized Return Quantiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Position-Return Scatter

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(8 * len(TICKERS), 5), sharey=False)

for ax, tk in zip(axes, sorted(id_to_ticker.values())):
    df_tk = bt[bt['ticker'] == tk].sample(n=min(5000, len(bt[bt['ticker'] == tk])),
                                           random_state=42)
    sc = ax.scatter(
        df_tk['position'], df_tk['fwd_return'],
        c=df_tk['strategy_ret'], cmap='RdYlGn',
        alpha=0.35, s=8, vmin=-0.003, vmax=0.003,
    )
    coefs = np.polyfit(df_tk['position'], df_tk['fwd_return'], 1)
    xs = np.linspace(df_tk['position'].min(), df_tk['position'].max(), 100)
    ax.plot(xs, np.polyval(coefs, xs), 'k--', lw=1.5, label=f'slope={coefs[0]:.4f}')
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.axvline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{tk} - Position vs Realized Return')
    ax.set_xlabel('Position')
    ax.set_ylabel('Realized Forward Return')
    ax.legend(fontsize=9)
    plt.colorbar(sc, ax=ax, label='Strategy Ret')

plt.suptitle('Position-Return Scatter (colour = strategy return)', fontsize=13)
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Trade-Level Analysis

In [ ]:
def extract_trades(df_ticker):
    """Extract trade records from a single-ticker DataFrame ordered by time."""
    df = df_ticker.sort_values('date').reset_index(drop=True)
    pos = df['position'].values
    ret = df['strategy_ret_tc'].values
    active = np.abs(pos) > TRADE_THRESH

    trades = []
    in_trade = False
    t_start = None
    t_sign  = 0
    t_ret   = []

    for i in range(len(pos)):
        sign_i = int(np.sign(pos[i]))
        if active[i]:
            if not in_trade or sign_i != t_sign:
                if in_trade and t_ret:
                    trades.append({'sign': t_sign, 'n_bars': len(t_ret),
                                   'gross_ret': sum(t_ret)})
                in_trade = True
                t_sign   = sign_i
                t_ret    = [ret[i]]
            else:
                t_ret.append(ret[i])
        else:
            if in_trade and t_ret:
                trades.append({'sign': t_sign, 'n_bars': len(t_ret),
                               'gross_ret': sum(t_ret)})
            in_trade = False
            t_ret    = []
            t_sign   = 0

    if in_trade and t_ret:
        trades.append({'sign': t_sign, 'n_bars': len(t_ret), 'gross_ret': sum(t_ret)})

    return pd.DataFrame(trades) if trades else pd.DataFrame(
        columns=['sign', 'n_bars', 'gross_ret'])

# -- Collect trades per ticker -------------------------------------------------
trade_rows = []
n_tickers = len(TICKERS)
fig, axes = plt.subplots(2, max(n_tickers, 1), figsize=(8 * max(n_tickers, 1), 10),
                         squeeze=False)

for col_idx, tk in enumerate(sorted(id_to_ticker.values())):
    df_tk = bt[bt['ticker'] == tk]
    trades = extract_trades(df_tk)
    if len(trades) == 0:
        trade_rows.append({'Ticker': tk, '# Trades': 0})
        continue

    wins  = trades['gross_ret'] > 0
    longs = trades['sign']  == 1
    row = {
        'Ticker'           : tk,
        '# Trades'         : len(trades),
        '# Long'           : int(longs.sum()),
        '# Short'          : int((~longs).sum()),
        'Avg Profit/Trade' : round(trades['gross_ret'].mean(), 6),
        'Long Avg Profit'  : round(trades.loc[longs,  'gross_ret'].mean(), 6) if longs.any() else 0,
        'Short Avg Profit' : round(trades.loc[~longs, 'gross_ret'].mean(), 6) if (~longs).any() else 0,
        'Win Rate (%)'     : round(wins.mean() * 100, 2),
        'Long Win (%)'     : round(wins[longs].mean()  * 100, 2) if longs.any()  else 0,
        'Short Win (%)'    : round(wins[~longs].mean() * 100, 2) if (~longs).any() else 0,
        'Avg Hold (bars)'  : round(trades['n_bars'].mean(), 1),
        'Best Trade'       : round(trades['gross_ret'].max(), 6),
        'Worst Trade'      : round(trades['gross_ret'].min(), 6),
    }
    trade_rows.append(row)

    colors = ['#27ae60' if v > 0 else '#e74c3c' for v in trades['gross_ret']]
    ax0 = axes[0, col_idx]
    ax0.bar(range(len(trades)), trades['gross_ret'].values,
            color=colors, alpha=0.75, width=1.0)
    ax0.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax0.set_title(f'{tk} OOS - Trade PnL (TC-adj)')
    ax0.set_xlabel('Trade #')
    ax0.set_ylabel('Gross Return')
    ax0.grid(True, alpha=0.25, axis='y')

    ax1 = axes[1, col_idx]
    ax1.hist(trades['n_bars'].values, bins=30,
             color='steelblue', alpha=0.8, edgecolor='white')
    ax1.axvline(trades['n_bars'].mean(), color='red', lw=1.5,
                linestyle='--', label=f'mean={trades["n_bars"].mean():.1f}')
    ax1.set_title(f'{tk} OOS - Holding Duration (bars)')
    ax1.set_xlabel('Bars Held')
    ax1.set_ylabel('Count')
    ax1.legend()
    ax1.grid(True, alpha=0.25)

df_trades = pd.DataFrame(trade_rows).set_index('Ticker')
print('\nTRADE-LEVEL ANALYSIS (OOS)')
print('=' * 90)
print(df_trades.to_string())

plt.suptitle('Trade-Level Analysis (OOS, TC-adjusted)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_trades.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Cumulative PnL & Rolling Sharpe

In [ ]:
ROLL_WINDOW = max(BARS_PER_DAY * 21, 200)

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=False)

palette_tk = {'CL': '#2980b9', 'HO': '#e67e22', 'RB': '#27ae60'}

# -- 1. Cumulative gross PnL ---------------------------------------------------
ax = axes[0]
for tk in sorted(id_to_ticker.values()):
    color = palette_tk.get(tk, 'gray')
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum   = df_tk['strategy_ret'].cumsum().values
    ax.plot(df_tk['date'].values, cum, lw=1.5, color=color, alpha=0.85, label=tk)

comb_daily = bt.groupby('date')['strategy_ret'].sum().sort_index().cumsum()
ax.plot(comb_daily.index, comb_daily.values, lw=2.2, color='black',
        linestyle='--', label='Combined')
ax.fill_between(comb_daily.index, comb_daily.values, 0,
                where=comb_daily.values >= 0, color='green', alpha=0.06)
ax.fill_between(comb_daily.index, comb_daily.values, 0,
                where=comb_daily.values < 0, color='red', alpha=0.06)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Cumulative Gross PnL (OOS)')
ax.set_ylabel('Cumulative PnL')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# -- 2. TC-adjusted cumulative PnL ---------------------------------------------
ax = axes[1]
for tk in sorted(id_to_ticker.values()):
    color = palette_tk.get(tk, 'gray')
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum   = df_tk['strategy_ret_tc'].cumsum().values
    ax.plot(df_tk['date'].values, cum, lw=1.5, color=color, alpha=0.85, label=f'{tk} (TC)')

comb_tc = bt.groupby('date')['strategy_ret_tc'].sum().sort_index().cumsum()
ax.plot(comb_tc.index, comb_tc.values, lw=2.2, color='black', linestyle='--',
        label='Combined (TC)')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title(f'Cumulative TC-Adjusted PnL  ({TC_COST_BPS} bps per leg)')
ax.set_ylabel('Cumulative PnL')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# -- 3. Rolling Sharpe (combined) ----------------------------------------------
ax = axes[2]
comb_sr  = bt.sort_values('date').groupby('date')['strategy_ret'].sum()
roll_mu  = comb_sr.rolling(ROLL_WINDOW, min_periods=BARS_PER_DAY * 5).mean()
roll_std = comb_sr.rolling(ROLL_WINDOW, min_periods=BARS_PER_DAY * 5).std() + 1e-8
roll_sh  = (roll_mu / roll_std) * ANNUALIZATION

ax.plot(roll_sh.index, roll_sh.values, color='steelblue', lw=1.2, label='Rolling Sharpe')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values >= 0, color='steelblue', alpha=0.15)
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values < 0, color='red', alpha=0.15)
ax.set_title(f'Rolling Annualised Sharpe (window={ROLL_WINDOW} bars)')
ax.set_ylabel('Sharpe')
ax.legend()
ax.grid(True, alpha=0.25)

plt.suptitle('HybridTCN OOS Backtest - Cumulative PnL & Rolling Sharpe',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_cum_pnl.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

def _drawdown_series(cum_series):
    run_max = cum_series.cummax()
    return run_max - cum_series

# -- Combined drawdown ---------------------------------------------------------
ax = axes[0]
dd = _drawdown_series(comb_tc)
ax.fill_between(dd.index, dd.values, color='#e74c3c', alpha=0.5)
ax.plot(dd.index, dd.values, color='#c0392b', lw=0.8)
ax.set_title(f'Combined Drawdown (TC-adj, OOS)   Max DD = {dd.max():.5f}')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.25)

# -- Per-ticker drawdowns ------------------------------------------------------
ax = axes[1]
for tk in sorted(id_to_ticker.values()):
    color = palette_tk.get(tk, 'gray')
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum_t = df_tk.set_index('date')['strategy_ret_tc'].cumsum()
    dd_t  = _drawdown_series(cum_t)
    ax.plot(dd_t.index, dd_t.values, color=color, lw=1.3, alpha=0.85,
            label=f'{tk} (max={dd_t.max():.5f})')
ax.set_title('Per-Ticker Drawdown (TC-adj, OOS)')
ax.set_ylabel('Drawdown')
ax.legend()
ax.grid(True, alpha=0.25)

plt.suptitle('Drawdown Analysis - HybridTCN (OOS)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Monthly PnL Calendar Heatmap

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS) + 1,
                          figsize=(7 * (len(TICKERS) + 1), 5))

slices = [('ALL', bt)] + [(tk, bt[bt['ticker'] == tk])
                           for tk in sorted(id_to_ticker.values())]

for ax, (label, df_s) in zip(axes, slices):
    df_s = df_s.copy()
    df_s['year']  = df_s['date'].dt.year
    df_s['month'] = df_s['date'].dt.month

    pivot = (
        df_s.groupby(['year', 'month'])['strategy_ret_tc']
        .sum()
        .unstack('month')
    )
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]
    pivot = pivot.reindex(columns=['Jan','Feb','Mar','Apr','May','Jun',
                                   'Jul','Aug','Sep','Oct','Nov','Dec'])

    abs_max = pivot.abs().max().max()
    sns.heatmap(
        pivot, ax=ax, cmap='RdYlGn',
        center=0, vmin=-abs_max, vmax=abs_max,
        annot=True, fmt='.4f', annot_kws={'size': 7},
        linewidths=0.5, cbar_kws={'shrink': 0.8},
    )
    ax.set_title(f'{label} - Monthly PnL (TC-adj)')
    ax.set_xlabel('')

plt.suptitle('Monthly PnL Calendar Heatmap - HybridTCN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_monthly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. Rolling Branch Weights vs Price vs Cumulative PnL

Three-panel plot per ticker:
1. **Rolling 3-branch weights** (TCN technical, sequential VPIN, spatial NB+VPIN) + regime weight
2. **Daily close price** — the underlying price context
3. **Cumulative PnL** (TC-adjusted)

In [ ]:
from CTAFlow.data.raw_formatting.intraday_manager import read_exported_df

# -- Load daily close prices for each ticker -----------------------------------
daily_close = {}
for tk in TICKERS:
    intra_path = DATA_ROOT / tk / 'intraday.csv'
    if intra_path.exists():
        df_intra = read_exported_df(intra_path)
        df_intra['date'] = df_intra.index.date
        daily_close[tk] = (
            df_intra.groupby('date')['Close']
            .last()
            .rename(tk)
        )
        daily_close[tk].index = pd.to_datetime(daily_close[tk].index)
        print(f'{tk}: loaded {len(daily_close[tk])} daily closes')

# -- Average branch weights per day, per ticker --------------------------------
daily_gates = (
    bt.groupby(['date', 'ticker'])
    .agg(
        gate_seq_mean=('gate_seq', 'mean'),
        gate_spatial_mean=('gate_spatial', 'mean'),
        regime_weight_mean=('regime_weight', 'mean'),
        w_tcn_mean=('w_tcn', 'mean'),
        w_seq_mean=('w_seq', 'mean'),
        w_spatial_mean=('w_spatial', 'mean'),
    )
    .reset_index()
)

ROLL_GATE_WINDOW = 10  # 10-day rolling average

fig, axes = plt.subplots(3, len(TICKERS), figsize=(7 * len(TICKERS), 14),
                         sharex='col')
if len(TICKERS) == 1:
    axes = axes.reshape(-1, 1)

for col_idx, tk in enumerate(sorted(id_to_ticker.values())):
    tk_gates = daily_gates[daily_gates['ticker'] == tk].set_index('date').sort_index()
    tk_gates_roll = tk_gates.rolling(ROLL_GATE_WINDOW, min_periods=1).mean()

    # -- Panel 1: Rolling 3-branch weights + regime weight ---------------------
    ax = axes[0, col_idx]
    ax.plot(tk_gates_roll.index, tk_gates_roll['w_tcn_mean'],
            color='#3498db', lw=1.5, label='TCN (Tech)', alpha=0.9)
    ax.plot(tk_gates_roll.index, tk_gates_roll['w_seq_mean'],
            color='#2ecc71', lw=1.5, label='Sequential (VPIN)', alpha=0.9)
    ax.plot(tk_gates_roll.index, tk_gates_roll['w_spatial_mean'],
            color='#e67e22', lw=1.5, label='Spatial (NB+VPIN)', alpha=0.9)
    ax.plot(tk_gates_roll.index, tk_gates_roll['regime_weight_mean'],
            color='#9b59b6', lw=1.2, linestyle='--', label='Regime Weight', alpha=0.7)

    # Stacked fill to show relative allocation
    ax.fill_between(tk_gates_roll.index, 0, tk_gates_roll['w_tcn_mean'],
                    alpha=0.08, color='#3498db')
    ax.fill_between(tk_gates_roll.index,
                    tk_gates_roll['w_tcn_mean'],
                    tk_gates_roll['w_tcn_mean'] + tk_gates_roll['w_seq_mean'],
                    alpha=0.08, color='#2ecc71')
    ax.set_title(f'{tk} - Rolling {ROLL_GATE_WINDOW}d Branch Weights')
    ax.set_ylabel('Relative Weight')
    ax.legend(fontsize=7, loc='upper right', ncol=2)
    ax.grid(True, alpha=0.25)
    ax.set_ylim(0, 1)

    # -- Panel 2: Daily close price --------------------------------------------
    ax = axes[1, col_idx]
    if tk in daily_close:
        close_oos = daily_close[tk].loc[
            (daily_close[tk].index >= bt['date'].min()) &
            (daily_close[tk].index <= bt['date'].max())
        ]
        ax.plot(close_oos.index, close_oos.values, color=palette_tk.get(tk, 'gray'),
                lw=1.2)
        ax.set_title(f'{tk} - Daily Close')
        ax.set_ylabel('Price')
    else:
        ax.set_title(f'{tk} - Daily Close (no data)')
    ax.grid(True, alpha=0.25)

    # -- Panel 3: Cumulative PnL (TC-adjusted) ---------------------------------
    ax = axes[2, col_idx]
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    daily_cum = df_tk.groupby('date')['strategy_ret_tc'].sum().sort_index().cumsum()
    ax.plot(daily_cum.index, daily_cum.values, color=palette_tk.get(tk, 'gray'),
            lw=1.5)
    ax.fill_between(daily_cum.index, daily_cum.values, 0,
                    where=daily_cum.values >= 0, color='green', alpha=0.08)
    ax.fill_between(daily_cum.index, daily_cum.values, 0,
                    where=daily_cum.values < 0, color='red', alpha=0.08)
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{tk} - Cumulative PnL (TC-adj)')
    ax.set_ylabel('Cumulative PnL')
    ax.grid(True, alpha=0.25)

plt.suptitle('HybridTCN - Rolling Branch Weights / Price / PnL (OOS)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_branch_weights_vs_price.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 16b. Combined Rolling Branch Weights (All Tickers)

Single figure with branch weight evolution, portfolio-level close, and combined PnL.

In [ ]:
# -- Daily average branch weights across ALL tickers ----------------------------
daily_gates_all = (
    bt.groupby('date')
    .agg(
        gate_seq_mean=('gate_seq', 'mean'),
        gate_spatial_mean=('gate_spatial', 'mean'),
        regime_weight_mean=('regime_weight', 'mean'),
        w_tcn_mean=('w_tcn', 'mean'),
        w_seq_mean=('w_seq', 'mean'),
        w_spatial_mean=('w_spatial', 'mean'),
    )
    .sort_index()
)
gates_roll_all = daily_gates_all.rolling(ROLL_GATE_WINDOW, min_periods=1).mean()

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=True)

# -- Panel 1: Rolling 3-branch weights (combined) -----------------------------
ax = axes[0]
ax.plot(gates_roll_all.index, gates_roll_all['w_tcn_mean'],
        color='#3498db', lw=1.8, label='TCN (Tech)')
ax.plot(gates_roll_all.index, gates_roll_all['w_seq_mean'],
        color='#2ecc71', lw=1.8, label='Sequential (VPIN)')
ax.plot(gates_roll_all.index, gates_roll_all['w_spatial_mean'],
        color='#e67e22', lw=1.8, label='Spatial (NB+VPIN)')
ax.plot(gates_roll_all.index, gates_roll_all['regime_weight_mean'],
        color='#9b59b6', lw=1.2, linestyle='--', label='Regime Weight')

# Stacked fill
ax.fill_between(gates_roll_all.index, 0, gates_roll_all['w_tcn_mean'],
                alpha=0.1, color='#3498db')
ax.fill_between(gates_roll_all.index,
                gates_roll_all['w_tcn_mean'],
                gates_roll_all['w_tcn_mean'] + gates_roll_all['w_seq_mean'],
                alpha=0.1, color='#2ecc71')
ax.fill_between(gates_roll_all.index,
                gates_roll_all['w_tcn_mean'] + gates_roll_all['w_seq_mean'],
                1.0,
                alpha=0.1, color='#e67e22')
ax.set_title(f'Rolling {ROLL_GATE_WINDOW}d Branch Weights (All Tickers)')
ax.set_ylabel('Relative Weight')
ax.legend(ncol=4, fontsize=9)
ax.grid(True, alpha=0.25)
ax.set_ylim(0, 1)

# -- Panel 2: Normalised daily close per ticker --------------------------------
ax = axes[1]
for tk in sorted(id_to_ticker.values()):
    if tk in daily_close:
        close_oos = daily_close[tk].loc[
            (daily_close[tk].index >= bt['date'].min()) &
            (daily_close[tk].index <= bt['date'].max())
        ]
        # Normalise to 100
        normed = close_oos / close_oos.iloc[0] * 100
        ax.plot(normed.index, normed.values, color=palette_tk.get(tk, 'gray'),
                lw=1.3, label=tk)
ax.set_title('Normalised Daily Close (base=100)')
ax.set_ylabel('Normalised Price')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# -- Panel 3: Combined cumulative PnL -----------------------------------------
ax = axes[2]
for tk in sorted(id_to_ticker.values()):
    color = palette_tk.get(tk, 'gray')
    tk_daily_cum = bt[bt['ticker'] == tk].groupby('date')['strategy_ret_tc'].sum().sort_index().cumsum()
    ax.plot(tk_daily_cum.index, tk_daily_cum.values, color=color, lw=1.2,
            alpha=0.7, label=tk)
ax.plot(comb_tc.index, comb_tc.values, lw=2.2, color='black', linestyle='--',
        label='Combined')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Cumulative PnL (TC-adj)')
ax.set_ylabel('Cumulative PnL')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

plt.suptitle('HybridTCN - Branch Weights / Price / PnL Dashboard (OOS)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 17. Gradient-Based Feature Importance

Since the HybridTCN's `_SeqEncoder` and `TCNBackbone` don't expose explicit
attention/gate weights at the feature level, we use **gradient x input**
attribution to measure per-feature importance:

1. **Tech features**: Which features in the tech lookback window drive position most?
2. **Sequential VPIN features**: Which VPIN columns matter for the sequential branch?
3. **Tech temporal**: Which bars in the lookback window matter (TCN temporal profile)?

In [ ]:
# -- Resolve feature names -----------------------------------------------------
# Use saved column names from before prep cleanup, or fall back to checkpoint
tech_cols = _tech_cols_saved if _tech_cols_saved else list(ckpt.get('tech_feature_cols', [f'tech_{i}' for i in range(F_TECH)]))
_seq_cols = _seq_cols_saved if _seq_cols_saved else [f'seq_{i}' for i in range(F_SEQ)]

print(f'Tech features ({len(tech_cols)}): {tech_cols[:10]}{"..." if len(tech_cols) > 10 else ""}')
print(f'Sequential VPIN features ({len(_seq_cols)}): {_seq_cols}')

In [ ]:
# -- Gradient x Input Attribution ----------------------------------------------
# We compute gradients of the model's position output w.r.t. tech_features and
# seq_vpin inputs to determine per-feature and per-timestep importance.

N_ATTR_BATCHES = min(20, len(full_loader))  # limit for speed

tech_feat_importance = np.zeros(len(tech_cols))       # (F_TECH,)
tech_temporal_importance = np.zeros(tech_lookback)    # (L,)
seq_feat_importance = np.zeros(len(_seq_cols))        # (F_SEQ,)
n_attr_samples = 0

# Per-ticker attribution
tech_feat_by_ticker = {tid: np.zeros(len(tech_cols)) for tid in id_to_ticker}
seq_feat_by_ticker = {tid: np.zeros(len(_seq_cols)) for tid in id_to_ticker}
ticker_counts = {tid: 0 for tid in id_to_ticker}

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

for batch_idx, batch in enumerate(full_loader):
    if batch_idx >= N_ATTR_BATCHES:
        break

    inputs, targets = unpack_v3_batch(batch, device=device)
    B = inputs['tech_features'].shape[0]

    # Enable gradients on inputs
    inputs['tech_features'] = inputs['tech_features'].detach().requires_grad_(True)
    if inputs.get('seq_vpin') is not None:
        inputs['seq_vpin'] = inputs['seq_vpin'].detach().requires_grad_(True)

    position, _ae_losses = model(**inputs, return_ae_losses=True)

    # Sum of absolute positions as scalar objective
    position.abs().sum().backward()

    # Gradient x Input for tech features: (B, L, F_TECH)
    tech_grad = inputs['tech_features'].grad
    if tech_grad is not None:
        attr_tech = (tech_grad * inputs['tech_features']).abs().detach().cpu().numpy()
        # Feature importance: average over batch and time
        tech_feat_importance += attr_tech.mean(axis=(0, 1)) * B
        # Temporal importance: average over batch and features
        tech_temporal_importance += attr_tech.mean(axis=(0, 2))[:tech_lookback] * B

        # Per-ticker
        tids = inputs['ticker_id'].cpu().numpy()
        for tid in id_to_ticker:
            mask = tids == tid
            if mask.sum() > 0:
                tech_feat_by_ticker[tid] += attr_tech[mask].mean(axis=(0, 1)) * mask.sum()
                ticker_counts[tid] += mask.sum()

    # Gradient x Input for seq_vpin: (B, S, F_SEQ)
    if inputs.get('seq_vpin') is not None and inputs['seq_vpin'].grad is not None:
        seq_grad = inputs['seq_vpin'].grad
        attr_seq = (seq_grad * inputs['seq_vpin']).abs().detach().cpu().numpy()
        seq_feat_importance += attr_seq.mean(axis=(0, 1)) * B

        tids = inputs['ticker_id'].cpu().numpy()
        for tid in id_to_ticker:
            mask = tids == tid
            if mask.sum() > 0:
                seq_feat_by_ticker[tid] += attr_seq[mask].mean(axis=(0, 1)) * mask.sum()

    n_attr_samples += B
    model.zero_grad()

# Normalise
tech_feat_importance /= n_attr_samples
tech_temporal_importance /= n_attr_samples
seq_feat_importance /= n_attr_samples

for tid in id_to_ticker:
    if ticker_counts[tid] > 0:
        tech_feat_by_ticker[tid] /= ticker_counts[tid]
        seq_feat_by_ticker[tid] /= ticker_counts[tid]

print(f'Attribution computed over {n_attr_samples:,} samples ({N_ATTR_BATCHES} batches)')

In [ ]:
# -- Visualise feature importance (4 panels) -----------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# -- 1. Top tech features (overall) --------------------------------------------
ax = axes[0, 0]
sort_idx = np.argsort(tech_feat_importance)[::-1]
n_show = min(25, len(tech_cols))
sorted_names = [tech_cols[i] for i in sort_idx[:n_show]]
sorted_vals  = tech_feat_importance[sort_idx[:n_show]]
# Normalise to percentages
sorted_vals_pct = sorted_vals / (tech_feat_importance.sum() + 1e-10) * 100

bars = ax.barh(range(n_show), sorted_vals_pct,
               color='#3498db', alpha=0.85, edgecolor='white')
ax.set_yticks(range(n_show))
ax.set_yticklabels(sorted_names, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Importance (%)')
ax.set_title(f'Tech Feature Importance (Grad x Input, top {n_show})')
ax.grid(True, alpha=0.25, axis='x')

# -- 2. Sequential feature importance ------------------------------------------
ax = axes[0, 1]
sort_idx_seq = np.argsort(seq_feat_importance)[::-1]
n_show_seq = min(20, len(_seq_cols))
sorted_seq_names = [_seq_cols[i] for i in sort_idx_seq[:n_show_seq]]
sorted_seq_vals  = seq_feat_importance[sort_idx_seq[:n_show_seq]]
sorted_seq_pct   = sorted_seq_vals / (seq_feat_importance.sum() + 1e-10) * 100

bars = ax.barh(range(n_show_seq), sorted_seq_pct,
               color='#2ecc71', alpha=0.85, edgecolor='white')
ax.set_yticks(range(n_show_seq))
ax.set_yticklabels(sorted_seq_names, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Importance (%)')
ax.set_title('Sequential VPIN Feature Importance (Grad x Input)')
ax.grid(True, alpha=0.25, axis='x')

# -- 3. Tech temporal profile (which bars in lookback matter) -------------------
ax = axes[1, 0]
L = len(tech_temporal_importance)
ax.bar(range(L), tech_temporal_importance, color='steelblue', alpha=0.8, width=1.0)
ax.set_xlabel(f'Tech Lookback Position (0=oldest, {L-1}=most recent)')
ax.set_ylabel('Mean |Grad x Input|')
ax.set_title(f'TCN Temporal Importance (which bars matter, L={L})')
ax.grid(True, alpha=0.25, axis='y')

# Add rolling average
if L > 5:
    roll_temp = pd.Series(tech_temporal_importance).rolling(5, min_periods=1).mean()
    ax.plot(range(L), roll_temp.values, color='red', lw=2.0, label='5-bar rolling avg')
    ax.legend()

# -- 4. Per-ticker tech feature importance (grouped bar) -----------------------
ax = axes[1, 1]
top_k = 12
top_features = [tech_cols[i] for i in np.argsort(tech_feat_importance)[::-1][:top_k]]
x_pos = np.arange(top_k)
width = 0.8 / max(len(TICKERS), 1)

for j, tid in enumerate(sorted(id_to_ticker)):
    tk = id_to_ticker[tid]
    tk_vals = np.array([tech_feat_by_ticker[tid][tech_cols.index(f)] for f in top_features])
    tk_vals_pct = tk_vals / (tech_feat_by_ticker[tid].sum() + 1e-10) * 100
    offset = (j - len(TICKERS)/2 + 0.5) * width
    ax.bar(x_pos + offset, tk_vals_pct, width=width, label=tk, alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(top_features, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Importance (%)')
ax.set_title(f'Tech Feature Importance by Ticker (top {top_k})')
ax.legend()
ax.grid(True, alpha=0.25, axis='y')

plt.suptitle('HybridTCN - Gradient-Based Feature Importance (OOS)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()

# -- Print tables --------------------------------------------------------------
print('\nTECH FEATURE IMPORTANCE (top 20):')
print(f'{"Feature":<35s} {"Importance":>12s} {"% of Total":>10s}')
print('-' * 60)
total_tech = tech_feat_importance.sum()
for i in np.argsort(tech_feat_importance)[::-1][:20]:
    pct = tech_feat_importance[i] / (total_tech + 1e-10) * 100
    print(f'  {tech_cols[i]:<33s} {tech_feat_importance[i]:>12.6f} {pct:>9.2f}%')

print(f'\nSEQUENTIAL VPIN FEATURE IMPORTANCE:')
print(f'{"Feature":<35s} {"Importance":>12s} {"% of Total":>10s}')
print('-' * 60)
total_seq = seq_feat_importance.sum()
for i in np.argsort(seq_feat_importance)[::-1]:
    pct = seq_feat_importance[i] / (total_seq + 1e-10) * 100
    print(f'  {_seq_cols[i]:<33s} {seq_feat_importance[i]:>12.6f} {pct:>9.2f}%')

## 17b. Per-Ticker Temporal Profile & Sequential Feature Heatmap

In [ ]:
# -- Per-ticker temporal attribution -------------------------------------------
# Re-run with per-ticker tracking
temporal_by_ticker = {tid: np.zeros(tech_lookback) for tid in id_to_ticker}
ticker_temporal_counts = {tid: 0 for tid in id_to_ticker}

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

for batch_idx, batch in enumerate(full_loader):
    if batch_idx >= N_ATTR_BATCHES:
        break

    inputs, targets = unpack_v3_batch(batch, device=device)
    inputs['tech_features'] = inputs['tech_features'].detach().requires_grad_(True)

    position, _ = model(**inputs, return_ae_losses=True)
    position.abs().sum().backward()

    tech_grad = inputs['tech_features'].grad
    if tech_grad is not None:
        attr_tech = (tech_grad * inputs['tech_features']).abs().detach().cpu().numpy()
        temporal_profile = attr_tech.mean(axis=2)  # (B, L)

        tids = inputs['ticker_id'].cpu().numpy()
        for tid in id_to_ticker:
            mask = tids == tid
            if mask.sum() > 0:
                temporal_by_ticker[tid] += temporal_profile[mask].sum(axis=0)[:tech_lookback]
                ticker_temporal_counts[tid] += mask.sum()

    model.zero_grad()

for tid in id_to_ticker:
    if ticker_temporal_counts[tid] > 0:
        temporal_by_ticker[tid] /= ticker_temporal_counts[tid]

# -- Plot per-ticker temporal + seq heatmap ------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Per-ticker temporal profile
ax = axes[0]
for tid in sorted(id_to_ticker):
    tk = id_to_ticker[tid]
    ax.plot(range(tech_lookback), temporal_by_ticker[tid],
            lw=1.5, label=tk, color=palette_tk.get(tk, 'gray'), alpha=0.85)
ax.set_xlabel(f'Tech Lookback Position (0=oldest, {tech_lookback-1}=most recent)')
ax.set_ylabel('Mean |Grad x Input|')
ax.set_title('TCN Temporal Profile by Ticker')
ax.legend()
ax.grid(True, alpha=0.25)

# 2. Sequential feature importance heatmap (ticker x feature)
ax = axes[1]
seq_hm_data = np.zeros((len(id_to_ticker), len(_seq_cols)))
tk_names = []
for row, tid in enumerate(sorted(id_to_ticker)):
    tk_names.append(id_to_ticker[tid])
    total = seq_feat_by_ticker[tid].sum() + 1e-10
    seq_hm_data[row] = seq_feat_by_ticker[tid] / total * 100

sns.heatmap(
    seq_hm_data, ax=ax,
    xticklabels=_seq_cols, yticklabels=tk_names,
    cmap='YlOrRd', annot=True, fmt='.1f', annot_kws={'size': 8},
    linewidths=0.5, cbar_kws={'label': '% Importance'},
)
ax.set_title('Sequential Feature Importance by Ticker (%)')
ax.set_xlabel('VPIN Feature')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.suptitle('HybridTCN - Per-Ticker Attribution Detail (OOS)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_per_ticker_attribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 18. Leakage Detection

In [ ]:
from scipy import stats

# -- Branch weight vs future return correlation --------------------------------
print('=' * 80)
print('LEAKAGE TEST SUMMARY')
print('=' * 80)

leak_results = []
for name, arr in [('Gate Seq', gate_seq_arr), ('Gate Spatial', gate_spatial_arr),
                  ('Regime Weight', regime_w_arr),
                  ('TCN Branch Weight', branch_weights_3[:, 0]),
                  ('Seq Branch Weight', branch_weights_3[:, 1]),
                  ('Spatial Branch Weight', branch_weights_3[:, 2])]:
    corr, pval = stats.pearsonr(arr, bt_ret)
    leak_results.append((name, corr, pval))
    rho, rho_p = stats.spearmanr(arr, bt_ret)
    leak_results.append((f'{name} (rank)', rho, rho_p))

leak_df = pd.DataFrame(leak_results, columns=['Branch', 'Correlation', 'p-value'])

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
colors_leak = ['#e74c3c' if abs(r) > 0.05 and p < 0.01 else '#27ae60'
               for r, p in zip(leak_df['Correlation'], leak_df['p-value'])]
ax.barh(range(len(leak_df)), leak_df['Correlation'].values,
        color=colors_leak, alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(leak_df)))
ax.set_yticklabels(leak_df['Branch'].values, fontsize=9)
ax.axvline(0, color='gray', lw=0.8, linestyle=':')
ax.axvline(0.05, color='red', lw=0.8, linestyle='--', alpha=0.5)
ax.axvline(-0.05, color='red', lw=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Pearson / Spearman Correlation with Future Return')
ax.set_title('Leakage Test: Gate & Branch Weights vs Fwd Return (OOS)')
ax.grid(True, alpha=0.25, axis='x')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{prefix}_bt_leakage.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'{"Branch":<30s} {"Corr":>8s} {"p-value":>10s} {"Flag":>6s}')
print('-' * 60)
for _, row in leak_df.iterrows():
    flag = 'LEAK?' if abs(row['Correlation']) > 0.05 and row['p-value'] < 0.01 else 'OK'
    print(f'  {row["Branch"]:<28s} {row["Correlation"]:>8.4f} {row["p-value"]:>10.2e} {flag:>6s}')

# -- Position correlation test (sanity) ----------------------------------------
pos_corr, pos_p = stats.pearsonr(bt_pos, bt_ret)
print(f'\nPosition vs Fwd Return: r={pos_corr:.4f}  p={pos_p:.2e}')
if pos_corr < 0:
    print('  WARNING: Negative position-return correlation -- model may be inverted!')
elif pos_corr > 0.15:
    print('  WARNING: Very high position-return correlation -- possible lookahead!')
else:
    print('  OK: Moderate positive correlation as expected.')

## 18b. Sequential Feature Leakage Check

In [ ]:
# -- Per-feature correlation with future return (spot check) -------------------
print('\n' + '=' * 80)
print('SEQUENTIAL FEATURE vs FWD RETURN CORRELATION (per-feature leakage check)')
print('=' * 80)
print('  Checking raw input features from the first batch for directional leakage...')

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

with torch.no_grad():
    for batch in full_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        seq_raw = inputs['seq_vpin'].cpu().numpy()  # (B, T, F_SEQ)
        seq_lens = inputs['seq_vpin_lens'].cpu().numpy()
        fwd_ret = targets.view(-1).float().cpu().numpy()
        break

n_samples, T, n_feat = seq_raw.shape
feat_means = np.zeros((n_samples, n_feat))
for i in range(n_samples):
    valid_len = int(seq_lens[i])
    if valid_len > 0:
        feat_means[i] = seq_raw[i, :valid_len, :].mean(axis=0)

print(f'\n{"Feature":<25s} {"Pearson r":>10s} {"p-value":>10s} {"Flag":>8s}')
print('-' * 60)
suspicious = []
for j in range(n_feat):
    fname = _seq_cols[j] if j < len(_seq_cols) else f'seq_{j}'
    r, p = stats.pearsonr(feat_means[:, j], fwd_ret)
    flag = 'SUSPECT' if abs(r) > 0.10 and p < 0.01 else ''
    if flag:
        suspicious.append(fname)
    print(f'  {fname:<23s} {r:>10.4f} {p:>10.2e} {flag:>8s}')

if suspicious:
    print(f'\n  SUSPICIOUS features (|r|>0.10, p<0.01): {suspicious}')
    print('  -> These features may contain forward-looking information!')
else:
    print('\n  No features show suspicious correlation with future returns.')

## 19. Export Results

In [ ]:
# -- Full backtest DataFrame ---------------------------------------------------
bt_export_path = BACKTEST_PATH / f'{prefix}_bt_full.csv'
bt.to_csv(bt_export_path, index=False)
print(f'Full backtest saved -> {bt_export_path}')

# -- Summary table -------------------------------------------------------------
summary_path = BACKTEST_PATH / f'{prefix}_bt_summary.csv'
df_summary.to_csv(summary_path)
print(f'Summary saved       -> {summary_path}')

# -- Trade-level table ---------------------------------------------------------
trades_path = BACKTEST_PATH / f'{prefix}_bt_trades.csv'
df_trades.to_csv(trades_path)
print(f'Trade table saved   -> {trades_path}')

# -- Feature importance --------------------------------------------------------
tech_imp_df = pd.DataFrame({
    'feature': tech_cols,
    'importance': tech_feat_importance,
    'pct': tech_feat_importance / (tech_feat_importance.sum() + 1e-10) * 100,
}).sort_values('importance', ascending=False)
tech_imp_df.to_csv(BACKTEST_PATH / f'{prefix}_bt_tech_importance.csv', index=False)

seq_imp_df = pd.DataFrame({
    'feature': _seq_cols,
    'importance': seq_feat_importance,
    'pct': seq_feat_importance / (seq_feat_importance.sum() + 1e-10) * 100,
}).sort_values('importance', ascending=False)
seq_imp_df.to_csv(BACKTEST_PATH / f'{prefix}_bt_seq_importance.csv', index=False)

# -- Daily branch weights -----------------------------------------------------
daily_gates_all.to_csv(BACKTEST_PATH / f'{prefix}_bt_daily_branch_weights.csv')

print(f'Feature importance saved')
print(f'Daily branch weights saved')

# -- Quick final print ---------------------------------------------------------
print('\n' + '=' * 70)
print(f'BACKTEST COMPLETE - OOS from {VAL_START_DATE}')
print('=' * 70)
for col_label, col in [('Gross', 'strategy_ret'), ('TC-adj', 'strategy_ret_tc')]:
    sr = bt[col].values
    ann_sh = (sr.mean() / (sr.std() + 1e-8)) * ANNUALIZATION
    print(f'  {col_label:8s}: Ann.Sharpe={ann_sh:.3f}   '
          f'Net PnL={sr.sum():.5f}   '
          f'Max DD={_metrics(bt, "", col=col)["Max Drawdown"]:.5f}')

print(f'\nMean branch weights (TCN / Seq / Spatial): '
      f'{bt["w_tcn"].mean():.3f} / {bt["w_seq"].mean():.3f} / {bt["w_spatial"].mean():.3f}')